# Bag of Words

**Lernziel:** Du baust einen Zählvektor in einer vorgegebenen Spaltenreihenfolge und bildest eine neue Review mit demselben Vokabular ab.

**Arbeitsauftrag · 10 Minuten**

1. Ergänze in Abschnitt 2 `vectorize()`. Die Funktion liefert pro Wort im Vokabular genau einen Zählwert.
2. Prüfe in Abschnitt 3, ob zwei verschiedene Reviews mit festem Vokabular denselben Vektor ergeben.

Zum Ausprobieren: In Abschnitt 4 wählst du eine Review-Datei aus und untersuchst ihren Vektor.

## 1. Beispieldaten und Wortzählung

Review A ist positiv, B ist negativ. `tokenize(text)` ersetzt `<br />` durch ein Leerzeichen, schreibt den Text klein und extrahiert Wörter mit `[a-z]+`. Die Funktion `count_words()` zählt jedes Token in einem Dictionary.

Das Vokabular entsteht aus den Tokens von A und B: `set()` sammelt jede Wortform einmal, `sorted()` ordnet die Wörter alphabetisch. Diese Reihenfolge legt die fünf Spalten der Matrix fest. Für C und D verwenden wir später dieselben Spalten.

In [ ]:
import re


def tokenize(text: str) -> list[str]:
    text = text.replace('<br />', ' ').lower()
    return re.findall(r'[a-z]+', text)


reference_reviews = [
    'Great acting. Great direction. Great soundtrack.',
    'Flat acting.',
]
vocabulary = sorted(set(tokenize(' '.join(reference_reviews))))

for review_id, review in zip(['A', 'B'], reference_reviews, strict=True):
    print(f'{review_id}: {review}')
print(f'Vocabulary: {vocabulary}')

In [ ]:
def count_words(tokens: list[str]) -> dict[str, int]:
    counts = {}
    for word in tokens:
        if word not in counts:
            counts[word] = 0
        counts[word] += 1
    return counts


def show_matrix(vocabulary: list[str], rows: list[list[int]], review_ids: list[str]) -> None:
    header = 'Review'
    print(f'{header:<8}' + ''.join(f'{word:>12}' for word in vocabulary))
    for review_id, row in zip(review_ids, rows, strict=True):
        print(f'{review_id:<8}' + ''.join(f'{value:>12}' for value in row))

## 2. Vom Review-Text zum Zählvektor

Implementiere `vectorize(text, vocabulary)`. Die Funktion erhält den vollständigen Review-Text.

1. Zerlege den Text mit `tokenize(text)` in Wörter.
2. Zähle die Wörter mit `count_words(tokens)`.
3. Gehe das Vokabular in seiner vorgegebenen Reihenfolge durch. Hänge für jedes Wort seine Häufigkeit an den Vektor an. Mit `counts.get(word, 0)` erhält ein fehlendes Wort den Wert null.

Gib den Vektor zurück und führe die Zelle mit der Matrix für A und B aus.

In [ ]:
def vectorize(text: str, vocabulary: list[str]) -> list[int]:
    raise NotImplementedError('Implement vectorize')

In [ ]:
reference_rows = [vectorize(text, vocabulary) for text in reference_reviews]
show_matrix(vocabulary, reference_rows, ['A', 'B'])

## 3. Festes Vokabular prüfen

Der erste Selbstcheck prüft ein Beispiel mit einer Wiederholung, einem unbekannten Wort und unsortiertem Vokabular. Sage dann voraus, ob `Great acting.` und `Great acting overall.` dieselbe Zeile ergeben. Führe die Zelle aus und begründe das Ergebnis am Vokabular.

In [ ]:
assert vectorize('Great great outside!', ['flat', 'great', 'acting']) == [0, 2, 0]
print('Vector check passed.')

short_vector = vectorize('Great acting.', vocabulary)
new_vector = vectorize('Great acting overall.', vocabulary)
show_matrix(vocabulary, [short_vector, new_vector], ['C', 'D'])
assert short_vector == new_vector

### Vergleich der vier Beispielreviews

Warum haben A, B, C und D dieselben fünf Spalten? Welche beiden Wörter kommen in D vor und stehen zugleich im Vokabular?

## 4. Zum Ausprobieren: Eine Review-Datei vektorisieren

Für die echten Reviews verwenden wir die Stems als Spalten. Die vorbereiteten Zellen lesen alle 100 Kursreviews ein und sortieren die unterschiedlichen Stems alphabetisch. Dieses Vokabular bleibt für jede ausgewählte Datei gleich.

Führe die Zellen der Reihe nach aus. Die Installationszelle brauchst du nur, wenn NLTK im aktuellen Kernel fehlt. Für den englischen Porter-Stemmer genügt das Paket.

In [ ]:
%pip install nltk==3.9.4

In [ ]:
from pathlib import Path
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
review_dir = Path('data/imdb/train')


def stem_review(text: str) -> list[str]:
    return [stemmer.stem(token) for token in tokenize(text)]


known_stems = set()
for path in review_dir.rglob('*.txt'):
    known_stems.update(stem_review(path.read_text(encoding='utf-8')))

review_vocabulary = sorted(known_stems)
print('Vocabulary size:', len(review_vocabulary))

### Wähle eine Review aus

Ändere `review_path` und führe die nächste Zelle erneut aus. Du kannst auch einen absoluten Pfad zu einer eigenen englischsprachigen UTF-8-Textdatei einsetzen. Neue Stems erhalten keine zusätzliche Spalte.

Auf den Originaltext und die gestemmten Tokens folgt der Vektor in kompakter Form. Die Tabelle zeigt die Position (ab 0), den Stem und den Zählwert jeder belegten Spalte. Alle übrigen Werte sind null. Den vollständigen Vektor kannst du mit `print(review_vector)` ansehen.

In [ ]:
review_path = review_dir / 'pos' / '10589_10.txt'
review_text = review_path.read_text(encoding='utf-8')
review_stems = stem_review(review_text)
review_vector = vectorize(' '.join(review_stems), review_vocabulary)

print('Original text:')
print(review_text)
print('\nStemmed tokens:')
print(' · '.join(review_stems))
print(f'\nVector ({len(review_vector)} columns):')
print('Index  Stem             Count')
for index, count in enumerate(review_vector):
    if count:
        print(f'{index:>5}  {review_vocabulary[index]:<16} {count:>5}')